# 04 — Recorte espacial e viabilidade de dados externos

Este notebook mede as duas coisas que `docs/04-dados-externos.md` deixou
condicionadas, e nada mais. Não modela.

**1. Expandir o recorte vale a pena?** D-16 coloca isso como a primeira coisa a
fazer, antes de qualquer fonte externa: é mais barato — o dado já está baixado e o
filtro é um parâmetro — e não introduz fonte de erro nova. O que se ganha é
variância territorial, e com ela a possibilidade de usar população municipal do
IBGE, que existe anualmente por município e dispensa mediação geográfica.

**2. Quem fica de fora da trilha geográfica?** D-17 obriga caracterizar os ~43%
não posicionáveis. Se eles diferirem sistematicamente em tipo ou gestão, a trilha
geográfica não fala sobre a rede, fala sobre um recorte enviesado dela.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import duckdb
import pandas as pd

from src import changes
from src.paths import PRIMARY_FOLDER

pd.set_option("display.width", 200)
con = duckdb.connect()
PERIODOS = changes.periodos_disponiveis()
ULTIMO = PERIODOS[-1]
SP = "355030"

raiz = lambda p: PRIMARY_FOLDER / p / "tbEstabelecimento.parquet"
print(f"snapshots: {PERIODOS}\nreferência para os cortes: {ULTIMO}")

## 1. Quanto cada recorte acrescenta

Três recortes possíveis, do mais estreito ao mais largo. O que interessa não é só
o número de nós: é quantos **municípios distintos** entram, porque é isso que dá
variância à população do IBGE.

A Região Metropolitana de São Paulo tem 39 municípios; os códigos IBGE de todos
começam com `35`, como todo o estado, então o recorte da RM precisa de lista
explícita.

In [ ]:
# Os 39 municípios da RMSP, códigos IBGE de 7 dígitos.
RMSP = [
    "3503901", "3505708", "3506607", "3509007", "3509205", "3510609", "3513009",
    "3513801", "3515004", "3515103", "3515707", "3516309", "3516408", "3518800",
    "3522208", "3522505", "3523107", "3525003", "3526209", "3528502", "3529401",
    "3530607", "3534401", "3539806", "3543303", "3544103", "3545001", "3546801",
    "3547304", "3547809", "3548708", "3548807", "3549904", "3550308", "3552502",
    "3552809", "3556453", "3557105", "3505500",
]
# O CNES grava co_municipio_gestor com 6 dígitos (sem o verificador).
RMSP6 = sorted({m[:6] for m in RMSP})

def perfil_recorte(nome: str, filtro: str) -> dict:
    r = con.execute(f'''
        SELECT COUNT(DISTINCT co_unidade) estabelecimentos,
               COUNT(DISTINCT co_municipio_gestor) municipios,
               COUNT(DISTINCT CASE WHEN nu_latitude IS NOT NULL
                                    AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                                   THEN co_unidade END) com_coordenada
        FROM read_parquet('{raiz(ULTIMO)}') WHERE {filtro}
    ''').fetchone()
    return {"recorte": nome, "estabelecimentos": r[0], "municipios": r[1],
            "com_coordenada": r[2],
            "cobertura_%": round(100 * r[2] / r[0], 1) if r[0] else 0.0}

lista_rm = ", ".join(f"'{m}'" for m in RMSP6)
recortes = pd.DataFrame([
    perfil_recorte("município de SP", f"co_municipio_gestor = '{SP}'"),
    perfil_recorte("RMSP (39 municípios)", f"co_municipio_gestor IN ({lista_rm})"),
    perfil_recorte("estado de SP", "co_municipio_gestor LIKE '35%'"),
    perfil_recorte("Brasil", "1 = 1"),
])
print(recortes.to_string(index=False))

### O que o recorte custa em tempo de execução

Expandir não é grátis: o grafo relacional cresce com o número de nós, e o espaço
de candidatos da tarefa de aquisição cresce com estabelecimentos × tipos de
equipamento. Medindo o tamanho do espaço de rótulos em cada recorte, para que a
decisão seja informada e não otimista.

In [ ]:
def tamanho_da_tarefa(nome: str, filtro: str) -> dict:
    equip = PRIMARY_FOLDER / ULTIMO / "rlEstabEquipamento.parquet"
    r = con.execute(f'''
        WITH sel AS (
            SELECT DISTINCT co_unidade FROM read_parquet('{raiz(ULTIMO)}')
            WHERE {filtro}
        ),
        itens AS (
            SELECT DISTINCT co_equipamento FROM read_parquet('{equip}')
            WHERE co_equipamento IS NOT NULL
        ),
        tem AS (
            SELECT DISTINCT co_unidade, co_equipamento
            FROM read_parquet('{equip}') JOIN sel USING (co_unidade)
        )
        SELECT (SELECT COUNT(*) FROM sel) AS estab,
               (SELECT COUNT(*) FROM itens) AS itens,
               (SELECT COUNT(*) FROM tem) AS pares_existentes
    ''').fetchone()
    return {"recorte": nome, "estabelecimentos": r[0], "tipos_equipamento": r[1],
            "pares_existentes": r[2],
            "candidatos_por_transicao": r[0] * r[1] - r[2]}

tamanhos = pd.DataFrame([
    tamanho_da_tarefa("município de SP", f"co_municipio_gestor = '{SP}'"),
    tamanho_da_tarefa("RMSP", f"co_municipio_gestor IN ({lista_rm})"),
    tamanho_da_tarefa("estado de SP", "co_municipio_gestor LIKE '35%'"),
])
tamanhos["candidatos_8_transicoes"] = tamanhos["candidatos_por_transicao"] * 8
print(tamanhos.to_string(index=False))
print("\nO espaço de candidatos é o custo dominante: ele multiplica por 8 transições")
print("e cada linha vira um exemplo de treino.")

> **Veredito — recorte espacial.**
>
> _(A RMSP acrescenta municípios suficientes para dar variância à população do
> IBGE, a um custo de candidatos que ainda caiba? Registrar a escolha e ajustar
> `MUNICIPIO_SAO_PAULO` / o parâmetro de recorte em `src/graph.py`.)_

## 2. Quem fica de fora da trilha geográfica

D-17 mede o teto de cobertura em 57% e obriga caracterizar a exclusão. Se os não
posicionáveis se distribuírem como os posicionáveis, a restrição custa poder
estatístico mas não introduz viés. Se não, a trilha 3 fala sobre um subconjunto e
isso precisa constar no reporte.

In [ ]:
ATRIBUTOS = ["tp_unidade", "tp_gestao", "nivel_dep", "tp_pfpj", "co_natureza_jur"]

def comparar_posicionaveis(filtro: str, atributo: str) -> pd.DataFrame:
    return con.execute(f'''
        WITH e AS (
            SELECT "{atributo}" AS valor,
                   (nu_latitude IS NOT NULL
                    AND NOT (nu_latitude = 0 AND nu_longitude = 0)) AS posicionavel
            FROM read_parquet('{raiz(ULTIMO)}') WHERE {filtro}
        )
        SELECT valor,
               SUM(CASE WHEN posicionavel THEN 1 ELSE 0 END) AS posicionavel,
               SUM(CASE WHEN posicionavel THEN 0 ELSE 1 END) AS sem_coordenada,
               COUNT(*) AS total
        FROM e GROUP BY valor ORDER BY total DESC
    ''').df()

filtro_sp = f"co_municipio_gestor = '{SP}'"
for atributo in ATRIBUTOS:
    df = comparar_posicionaveis(filtro_sp, atributo)
    if df.empty:
        continue
    df["cobertura_%"] = (100 * df["posicionavel"] / df["total"]).round(1)
    base = 100 * df["posicionavel"].sum() / df["total"].sum()
    df["desvio_pp"] = (df["cobertura_%"] - base).round(1)
    print(f"\n{'=' * 72}\n{atributo}  (cobertura média = {base:.1f}%)\n{'=' * 72}")
    print(df.head(12).to_string(index=False))

In [ ]:
# Teste formal: a cobertura depende do atributo, ou é uniforme?
from scipy.stats import chi2_contingency

print(f"{'atributo':22} {'chi2':>12} {'p':>12} {'V de Cramer':>12}")
for atributo in ATRIBUTOS:
    df = comparar_posicionaveis(filtro_sp, atributo)
    df = df[(df["total"] >= 20) & df["valor"].notna()]
    if len(df) < 2:
        continue
    tabela = df[["posicionavel", "sem_coordenada"]].to_numpy()
    chi2, p, _, _ = chi2_contingency(tabela)
    n = tabela.sum()
    cramer = (chi2 / (n * (min(tabela.shape) - 1))) ** 0.5
    print(f"{atributo:22} {chi2:>12.1f} {p:>12.2e} {cramer:>12.3f}")

print("\nV de Cramer perto de 0 = cobertura uniforme, exclusão só custa poder.")
print("V alto = a exclusão é seletiva e a trilha 3 fala de um subconjunto enviesado.")

> **Veredito — viés de posicionabilidade.**
>
> _(A exclusão é uniforme ou seletiva? Se seletiva, em qual atributo e com que
> força? Registrar em D-17 e incluir a ressalva no reporte da trilha 3.)_

## 3. Pré-requisito da população do IBGE

D-16 libera a população municipal **se** o recorte expandir. O que falta verificar
é trivial mas não pode ser suposto: o código de município do CNES casa com o
código do IBGE?

O CNES grava `co_municipio_gestor` com 6 dígitos; o IBGE usa 7, sendo o último um
dígito verificador. A junção é determinística — truncar o código do IBGE — mas a
cobertura precisa ser medida, não assumida.

In [ ]:
r = con.execute(f'''
    SELECT COUNT(DISTINCT co_municipio_gestor) municipios,
           SUM(CASE WHEN LENGTH(co_municipio_gestor) = 6 THEN 1 ELSE 0 END) com_6,
           SUM(CASE WHEN LENGTH(co_municipio_gestor) <> 6 THEN 1 ELSE 0 END) outro,
           COUNT(*) linhas
    FROM read_parquet('{raiz(ULTIMO)}') WHERE co_municipio_gestor LIKE '35%'
''').df()
print("Estado de SP, formato de co_municipio_gestor:")
print(r.to_string(index=False))

# O IBGE tem 645 municípios em SP. Quantos aparecem no CNES?
n = int(r["municipios"].iloc[0])
print(f"\nmunicípios distintos no CNES: {n} (IBGE registra 645 em SP)")
print("Se a contagem bater, a junção por truncamento do código IBGE é direta;")
print("qualquer divergência precisa ser listada antes de a população entrar.")

> **Veredito — chave do IBGE.**
>
> _(O código casa? Quantos municípios divergem? Se a junção for limpa, D-16
> item 2 fica desbloqueado assim que o recorte expandir.)_

## Decisão

| Item | Veredito | Consequência no código |
|---|---|---|
| Recorte espacial | | parâmetro de recorte em `src/graph.py` |
| Viés de posicionabilidade | | ressalva no reporte da trilha 3, D-17 |
| Chave do IBGE | | desbloqueia D-16 item 2 |

O que **não** se decide aqui: SIA/SUS. Aquele teste exige baixar um mês de
produção ambulatorial e medir pareamento por `co_cnes` — trabalho próprio, com
dependência nova, descrito na seção 3.2 de `docs/04-dados-externos.md`.